# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sorgerator/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

*   **Selected Lane:** Content Refresh Lane
*   **ML Task Type:** Scoring / Regression
*   **Problem Framing:** 
    Instead of predicting a flat binary category (e.g., hardcoded "Needs Refresh" vs "Healthy"), this task is framed as a continuous scoring problem. The model outputs an urgency/priority score bounded between $0.0$ and $1.0$. 
*   **Downstream Action:** 
    A score of $0.0$ represents a highly stable or growing page, while a score approaching $1.0$ flags severe performance drop-off. This continuous output allows us to dynamically sort and rank URLs to generate a fluid, prioritized review queue for the editorial team, ensuring limited writing resources are always directed to the highest-decay assets first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

*   **The Ideal Target:** 
    The true structural "staleness" or qualitative decay of a page's content (i.e., information that is outdated, thin, or missing critical user intent signals).
*   **The Proxy Label:** 
    Because true content quality cannot be directly extracted from search logs or databases on day one, we use a continuous **Traffic and Impression Decay Rate** as our mathematical proxy label.
*   **Formula:** 
    We calculate the percentage drop in organic impressions by comparing recent performance against a historical baseline or peak window:
    
    $$\text{Decay Rate} = 1.0 - \left( \frac{\text{Recent Impressions}}{\text{Historical Peak Impressions}} \right)$$

*   **Proxy Limitations & Assumptions:** 
    While a steep drop in impressions strongly correlates with content that has lost its competitive edge in search results, it is a proxy, not a perfect measurement. Traffic drops can occasionally be driven by external macroeconomic shifts, algorithmic updates, or seasonal trends (e.g., a skiing article naturally losing visibility in July). The downstream model will rely on multi-signal feature combinations to help separate true content stale-out from external seasonal noise.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

*   **Product Success Metric:** 
    The primary business objective is to halt search traffic decay and recover lost visibility. Product success is measured by the average increase in organic search impressions and clicks for a URL within 6 weeks *after* it undergoes an editorial refresh from our prioritized queue, compared to its pre-refresh decay baseline.

*   **Separation from Model Metrics:** 
    We must explicitly separate product success from model evaluation performance:
    *   **Model Metrics:** Technical health indicators like Mean Absolute Error ($MAE$) or Mean Squared Error ($MSE$) evaluate how accurately the model predicts the numerical decay rate during training.
    *   **Product Metrics:** Real-world validation of the tool's utility. A model can achieve a near-perfect historical error rate, but if the content team ignores the queue, or if rewriting the flagged pages fails to stop the organic traffic slide, the framework delivers zero business value.

*   **Definition of Failure:** 
    The system is failing if the editorial team consistently updates high-priority pages from our queue over a two-month period, yet those URLs continue to lose search traction at their historical rate. This outcome would signal either a flawed proxy label (the decay footprint does not map to true content rot) or severe feature noise causing the model to prioritize the wrong assets.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

*   **Definition:** 
    One row in our dataset represents **one unique URL (web page) for a specific domain snapshot**. It captures the aggregated organic search performance metrics and structural signals of that single page over our baseline monitoring window.

*   **The Content Action:** 
    By setting the individual URL as our fundamental unit of analysis, the model's scoring output translates directly into a tangible editorial action. A high decay score maps directly to a specific webpage, allowing us to dynamically route that URL to a content writer for a refresh.

*   **Data Verification:**
    The code execution cell below pulls a small slice of `data/raw/content_refresh_anonymized.csv` to visually verify this structure using unique anonymized content IDs and their baseline search performance metrics.

In [6]:
import pandas as pd

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape} (Rows, Columns)")

sample_columns = ['content_id', 'impressions_90d', 'clicks_90d', 'avg_position']
df[sample_columns].head(5)

Dataset Shape: (30000, 44) (Rows, Columns)


,content_id,impressions_90d,clicks_90d,avg_position
0,content_304f48230142,3803,29,10.6
1,content_a1fb4e703a9e,15320,7,20.3
2,content_9aa793d4d895,12581,11,36.5
3,content_331d6c4de07b,11751,58,6.2
4,content_d99b7a2d90ca,19140,24,44.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

*   **The Heuristic Approach:** 
    A standard, non-ML approach to building a refresh queue would be writing a hardcoded SQL or Python rule, such as: 
    `IF impressions drop by > 20% AND content_age_days > 365 THEN flag_for_refresh`.

*   **Why Heuristics Fail Here:**
    1.  **Brittle Thresholds:** A 20% drop means something completely different for a high-traffic page (losing 20,000 impressions) versus a low-traffic page (losing 10 impressions). Hardcoded rules fail to capture scale and magnitude smoothly.
    2.  **Blind to Seasonality & Context:** A rigid threshold cannot distinguish between an article that is genuinely rotting and an article experiencing a normal seasonal dip. It treats all traffic drops as identical emergencies.
    3.  **Dimensionality Limits:** Our dataset contains over 40 columns (including `competition`, `cpc`, `avg_position`, and `scroll_rate`). Writing nested `if/else` statements to weigh how all these different signals interact would instantly become an unmaintainable nightmare.

*   **The ML Advantage:**
    By using a continuous Machine Learning scoring model, the system can map non-linear, multi-dimensional relationships. It learns to weigh dozens of features simultaneously—recognizing, for example, that a slight impression drop on a high-competition page is actually more urgent than a massive drop on a low-volume, seasonal page. ML gives us a dynamic, adaptive priority queue without forcing engineers to manually guess and hardcode arbitrary thresholds.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.